# 01 – Data Ingestion & Cleaning
### AI-Powered Predictive Bed Demand Forecasting — Albion Care Network

**Notebook 1 of 7** | Data Foundation Phase

---

## Objectives

1. Ingest all six raw operational datasets supplied by Albion Care Network.
2. Profile each dataset: shape, schema, missingness, duplicates, and logical/referential integrity.
3. Resolve data quality issues identified during profiling (inconsistent text casing, duplicate
   records, invalid timestamps, missing values) using justified, documented rules.
4. Reconcile identifiers (hospitals, wards, bed types, specialties) so every dataset shares a
   single, consistent vocabulary that later notebooks can join on.
5. Persist clean, analysis-ready datasets to `data/processed/` for use in Notebooks 02–07.

## Background

Albion Care Network's operational data is fragmented across five source systems (ADT/admissions,
bed management, ED/outpatient, theatre scheduling, and workforce), plus a static hospital reference
table. Each system was extracted independently, and — as is typical of real hospital data feeds —
free-text categorical fields were entered inconsistently across shifts and sites (e.g. `"ICU"`,
`"icu"`, `" Icu "` all referring to the same ward). Before any forecasting work can begin, we need
a single, trustworthy, de-duplicated version of each dataset with consistent keys.

## Inputs

| File | Rows (raw) | Grain |
|---|---|---|
| `hospital_reference.csv` | 5 | 1 row per hospital |
| `admissions_discharges.csv` | ~131k | 1 row per admission episode |
| `bed_inventory_occupancy.csv` | ~702k | 1 row per hospital–ward–bed_type–hour |
| `ed_outpatient_arrivals.csv` | ~362k | 1 row per ED/outpatient arrival |
| `elective_surgery_schedule.csv` | ~25k | 1 row per scheduled surgery |
| `staffing_resource.csv` | ~117k | 1 row per hospital–ward–role–day |

## Outputs

Six cleaned files in `data/processed/`, a data dictionary, and a data-quality log documenting
every rule applied (so the cleaning is auditable by both technical and business stakeholders).


In [1]:
# --- Setup & configuration -------------------------------------------------
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

RAW_DIR = Path('../data/raw')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

# A running log of every cleaning decision, so the notebook is fully auditable.
quality_log = []

def log_step(dataset, issue, action, rows_affected):
    quality_log.append({
        'dataset': dataset,
        'issue': issue,
        'action_taken': action,
        'rows_affected': rows_affected
    })

print('Environment ready. Raw data directory:', RAW_DIR.resolve())


Environment ready. Raw data directory: C:\Users\ifech\OneDrive\Desktop\Albion Care Network\hospital_occupancy_forecast\data\raw


In [2]:
def normalize_categorical(series: pd.Series) -> pd.Series:
    """Standardise free-text categorical values: trim whitespace, collapse
    internal double-spaces, apply Title Case, then restore known acronyms
    (e.g. ICU) that Title Case would otherwise mangle."""
    s = series.astype('string')
    s = s.str.strip()
    s = s.str.replace(r'\s+', ' ', regex=True)
    s = s.str.title()
    s = s.str.replace(r'\bIcu\b', 'ICU', regex=True)
    return s

def profile_dataset(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Return a one-row-per-column profiling summary for quick data-quality review."""
    rows = []
    for c in df.columns:
        col = df[c]
        rows.append({
            'column': c,
            'dtype': str(col.dtype),
            'n_missing': col.isna().sum(),
            'pct_missing': round(100 * col.isna().mean(), 2),
            'n_unique': col.nunique(dropna=True)
        })
    out = pd.DataFrame(rows)
    print(f'--- {name}: {df.shape[0]:,} rows x {df.shape[1]} columns ---')
    return out


---
## Section 1 – Hospital Reference (Dimension Table)

The smallest dataset, but the most important for joins: it defines every valid `hospital_id`
and gives business-friendly attributes (name, city, tier, specialty emphasis, capacity) that
we'll attach to every other dataset for readable dashboards later.


In [3]:
hospital_ref = pd.read_csv(RAW_DIR / 'hospital_reference.csv')
display(hospital_ref)
display(profile_dataset(hospital_ref, 'hospital_reference'))


,hospital_id,hospital_name,city,tier,specialty_emphasis,total_bed_capacity
0,HHN-LON-01,Horizon London Central,London,flagship,general,224
1,HHN-LON-02,Horizon Riverside,London,large,oncology,152
2,HHN-MAN-01,Horizon Manchester,Manchester,large,orthopaedics,164
3,HHN-BIR-01,Horizon Birmingham,Birmingham,medium,cardiology,126
4,HHN-EDI-01,Horizon Edinburgh,Edinburgh,medium,general,94


--- hospital_reference: 5 rows x 6 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,hospital_id,str,0,0.00,5
1,hospital_name,str,0,0.00,5
2,city,str,0,0.00,4
3,tier,str,0,0.00,3
4,specialty_emphasis,str,0,0.00,4
5,total_bed_capacity,int64,0,0.00,5


In [4]:
# hospital_reference is already clean (no missing values, small controlled vocabulary).
# We just lock in the valid hospital_id set for referential-integrity checks later.
VALID_HOSPITAL_IDS = set(hospital_ref['hospital_id'])
print('Valid hospital IDs:', sorted(VALID_HOSPITAL_IDS))

hospital_ref.to_parquet(PROC_DIR / 'hospital_reference_clean.parquet', index=False)
hospital_ref.to_csv(PROC_DIR / 'hospital_reference_clean.csv', index=False)


Valid hospital IDs: ['HHN-BIR-01', 'HHN-EDI-01', 'HHN-LON-01', 'HHN-LON-02', 'HHN-MAN-01']


---
## Section 2 – Admissions & Discharges

This is the core clinical-flow dataset: one row per patient admission episode, with the
admission/discharge timestamps, specialty, ward, bed type and length of stay that drive bed
demand. Profiling below.


In [5]:
adm_raw = pd.read_csv(RAW_DIR / 'admissions_discharges.csv')
display(adm_raw.head())
display(profile_dataset(adm_raw, 'admissions_discharges (raw)'))


,hospital_id,patient_id,admission_id,admission_datetime,discharge_datetime,admission_type,admission_source,specialty,ward,bed_type,length_of_stay_hours,discharge_destination
0,HHN-MAN-01,PAT0316166,ADM00063791,2024-12-03 11:59:00,2024-12-06 18:35:00,Emergency,Emergency Department,Orthopaedics,Orthopaedics Ward B,Standard,78.60,Home with Care Package
1,HHN-BIR-01,PAT0292869,ADM00081326,2024-07-13 14:21:00,2024-07-17 00:15:00,Emergency,Emergency Department,Orthopaedics,Orthopaedics Ward B,Standard,81.90,Rehabilitation Facility
2,HHN-LON-02,PAT0073285,ADM00047712,2025-06-24 07:36:00,2025-06-30 14:24:00,Emergency,Emergency Department,Orthopaedics,Orthopaedics Ward B,Standard,150.80,Home
3,HHN-LON-01,PAT0262803,ADM00014431,2024-12-09 15:16:00,2024-12-11 00:34:00,Emergency,Emergency Department,Orthopaedics,Orthopaedics Ward A,Standard,33.30,Home
4,HHN-LON-02,PAT0181895,ADM00118155,2025-08-20 06:42:00,2025-08-24 09:08:00,Elective,Waiting List,Orthopaedics,Orthopaedics Ward A,ICU Step-down,98.40,Home


--- admissions_discharges (raw): 131,479 rows x 12 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,hospital_id,str,0,0.00,5
1,patient_id,str,0,0.00,114522
2,admission_id,str,0,0.00,130956
3,admission_datetime,str,0,0.00,121310
4,discharge_datetime,str,0,0.00,123027
5,admission_type,str,0,0.00,3
6,admission_source,str,0,0.00,2
7,specialty,str,0,0.00,23
8,ward,str,0,0.00,31
9,bed_type,str,0,0.00,4


### 2.1 Data quality findings

Profiling the raw file surfaces several issues that need explicit, business-justified fixes:

1. **Inconsistent text casing/whitespace** in `specialty`, `ward`, `discharge_destination`
   (e.g. `'ICU'`, `'icu'`, `' Icu '` all present as distinct values) — this would silently
   fragment groupby/forecast keys if left uncleaned.
2. **Duplicate records** — some `admission_id`s appear more than once, including exact
   full-row duplicates (likely re-extraction artefacts) and near-duplicates that differ only
   by a missing `discharge_destination`.
3. **Missing `length_of_stay_hours`** (~1.8% of rows) — can often be recomputed directly from
   `admission_datetime`/`discharge_datetime` rather than dropped or blindly imputed.
4. **Discharge before admission** — a small number of rows have `discharge_datetime` earlier
   than `admission_datetime`, which is logically impossible and indicates a data-entry error.
5. **Missing `discharge_destination`** (~2.5%) — genuinely unknown at time of export for some
   still-open episodes; we retain these as `Undisclosed` rather than dropping the episode.

Each is investigated and resolved below, in order.


In [6]:
# --- Issue 1: inconsistent categorical text -------------------------------
before_specialty = adm_raw['specialty'].nunique()
before_ward = adm_raw['ward'].nunique()
before_dest = adm_raw['discharge_destination'].nunique()

adm = adm_raw.copy()
for col in ['specialty', 'ward', 'discharge_destination', 'admission_type',
            'admission_source', 'bed_type']:
    adm[col] = normalize_categorical(adm[col])

print(f"specialty: {before_specialty} raw values -> {adm['specialty'].nunique()} clean values")
print(f"ward: {before_ward} raw values -> {adm['ward'].nunique()} clean values")
print(f"discharge_destination: {before_dest} raw values -> {adm['discharge_destination'].nunique()} clean values")
print()
print('Clean specialty values:', sorted(adm['specialty'].dropna().unique()))
print('Clean ward values:', sorted(adm['ward'].dropna().unique()))

log_step('admissions_discharges', 'Inconsistent casing/whitespace in specialty/ward/discharge_destination',
         'Trimmed, collapsed whitespace, Title-Cased, restored ICU acronym',
         len(adm_raw))


specialty: 23 raw values -> 6 clean values
ward: 31 raw values -> 8 clean values
discharge_destination: 24 raw values -> 6 clean values

Clean specialty values: ['Cardiology', 'Diagnostics', 'General Medicine', 'ICU', 'Oncology', 'Orthopaedics']
Clean ward values: ['Cardiology Ward', 'Day Case Unit', 'General Medicine Ward A', 'General Medicine Ward B', 'ICU', 'Oncology Ward', 'Orthopaedics Ward A', 'Orthopaedics Ward B']


In [7]:
# --- Issue 2: duplicate records --------------------------------------------
n_exact_dupes = adm.duplicated().sum()
print(f'Exact full-row duplicates: {n_exact_dupes}')
adm = adm.drop_duplicates().copy()
log_step('admissions_discharges', 'Exact full-row duplicate records', 'Dropped duplicate rows', n_exact_dupes)

# Remaining admission_id collisions after exact-dupe removal are near-duplicates that
# differ only by a missing discharge_destination in one copy. Keep the more complete row.
remaining_dupe_ids = adm[adm.duplicated(subset=['admission_id'], keep=False)]
print(f'Remaining admission_id collisions after exact-dupe removal: {len(remaining_dupe_ids)} rows')

adm = adm.sort_values('discharge_destination', na_position='last')
n_before = len(adm)
adm = adm.drop_duplicates(subset=['admission_id'], keep='first')
n_dropped = n_before - len(adm)
print(f'Dropped {n_dropped} incomplete near-duplicate rows, keeping the most complete record per admission_id')
log_step('admissions_discharges', 'Residual admission_id duplicates with partial missingness',
         'Kept the more complete row per admission_id (preferring non-null discharge_destination)', n_dropped)

assert adm['admission_id'].is_unique, 'admission_id should now be unique'
print('admission_id is now unique:', adm['admission_id'].is_unique)


Exact full-row duplicates: 486
Remaining admission_id collisions after exact-dupe removal: 74 rows
Dropped 37 incomplete near-duplicate rows, keeping the most complete record per admission_id
admission_id is now unique: True


In [8]:
# --- Issue 3 & 4: timestamps, recomputed LOS, and impossible date ordering --
adm['admission_datetime'] = pd.to_datetime(adm['admission_datetime'])
adm['discharge_datetime'] = pd.to_datetime(adm['discharge_datetime'], errors='coerce')

# Logically impossible rows: discharge strictly before admission.
bad_order = adm['discharge_datetime'] < adm['admission_datetime']
print(f'Rows with discharge_datetime before admission_datetime: {bad_order.sum()}')
log_step('admissions_discharges', 'discharge_datetime earlier than admission_datetime (data-entry error)',
         'Dropped affected rows (timestamps cannot be trusted for these episodes)', int(bad_order.sum()))
adm = adm.loc[~bad_order].copy()

# Recompute length_of_stay_hours from timestamps wherever both are present; this both
# fills genuine gaps and corrects any stored LOS values that drifted from the timestamps.
recomputed_los = (adm['discharge_datetime'] - adm['admission_datetime']).dt.total_seconds() / 3600
missing_los = adm['length_of_stay_hours'].isna()
mismatched_los = (~missing_los) & ((recomputed_los - adm['length_of_stay_hours']).abs() > 1)

print(f'Missing length_of_stay_hours (recomputed from timestamps): {missing_los.sum()}')
print(f'Stored length_of_stay_hours inconsistent with timestamps by >1h (recomputed): {mismatched_los.sum()}')

adm.loc[missing_los | mismatched_los, 'length_of_stay_hours'] = recomputed_los.loc[missing_los | mismatched_los]
log_step('admissions_discharges', 'Missing length_of_stay_hours', 'Recomputed from admission/discharge timestamps', int(missing_los.sum()))
log_step('admissions_discharges', 'length_of_stay_hours inconsistent with timestamps', 'Overwritten with recomputed value from timestamps', int(mismatched_los.sum()))

# Any still-open episodes with no discharge_datetime keep length_of_stay_hours as recorded
# (can't recompute) -- flag them explicitly as "still admitted" for downstream notebooks.
adm['is_open_episode'] = adm['discharge_datetime'].isna()
print(f'Still-open episodes (no discharge recorded): {adm["is_open_episode"].sum()}')


Rows with discharge_datetime before admission_datetime: 196
Missing length_of_stay_hours (recomputed from timestamps): 2352
Stored length_of_stay_hours inconsistent with timestamps by >1h (recomputed): 0
Still-open episodes (no discharge recorded): 0


In [9]:
# --- Issue 5: missing discharge_destination ---------------------------------
missing_dest = adm['discharge_destination'].isna() & ~adm['is_open_episode']
print(f'Discharged episodes with missing discharge_destination: {missing_dest.sum()}')
adm['discharge_destination'] = adm['discharge_destination'].fillna('Undisclosed')
log_step('admissions_discharges', 'Missing discharge_destination for discharged episodes',
         "Filled with explicit 'Undisclosed' category (destination genuinely not captured at source)", int(missing_dest.sum()))


Discharged episodes with missing discharge_destination: 3246


In [10]:
# --- Referential integrity check against hospital_reference -----------------
invalid_hosp = ~adm['hospital_id'].isin(VALID_HOSPITAL_IDS)
print(f'Admission rows with an unrecognised hospital_id: {invalid_hosp.sum()}')

print('\nFinal admissions_discharges profile:')
display(profile_dataset(adm, 'admissions_discharges (clean)'))
adm.describe(include='all').T.head(15)


Admission rows with an unrecognised hospital_id: 0

Final admissions_discharges profile:
--- admissions_discharges (clean): 130,760 rows x 13 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,hospital_id,str,0,0.00,5
1,patient_id,str,0,0.00,114380
2,admission_id,str,0,0.00,130760
3,admission_datetime,datetime64[us],0,0.00,121139
4,discharge_datetime,datetime64[us],0,0.00,122857
5,admission_type,string,0,0.00,3
6,admission_source,string,0,0.00,2
7,specialty,string,0,0.00,6
8,ward,string,0,0.00,8
9,bed_type,string,0,0.00,4


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
hospital_id,130760,5,HHN-LON-01,37974,NaN,NaN,NaN,NaN,NaN,NaN,NaN
patient_id,130760,114380,PAT0444549,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_id,130760,130760,ADM00097351,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_datetime,130760,NaN,NaN,NaN,2024-12-27 05:23:28.627562,2024-01-01 01:29:00,2024-06-21 13:12:00,2024-12-31 09:18:00,2025-06-23 08:34:15,2026-01-01 03:16:00,NaN
discharge_datetime,130760,NaN,NaN,NaN,2024-12-30 16:35:18.891098,2024-01-01 15:57:00,2024-06-25 10:43:15,2025-01-03 19:14:00,2025-06-26 16:29:30,2026-02-19 07:13:00,NaN
admission_type,130760,3,Emergency,104447,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_source,130760,2,Emergency Department,107431,NaN,NaN,NaN,NaN,NaN,NaN,NaN
specialty,130760,6,General Medicine,46276,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ward,130760,8,Cardiology Ward,24646,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bed_type,130760,4,Standard,113892,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
adm.to_parquet(PROC_DIR / 'admissions_discharges_clean.parquet', index=False)
adm.to_csv(PROC_DIR / 'admissions_discharges_clean.csv', index=False)
print(f'Saved {len(adm):,} clean admission records ({adm_raw.shape[0]:,} -> {adm.shape[0]:,} rows after cleaning).')


Saved 130,760 clean admission records (131,479 -> 130,760 rows after cleaning).


---
## Section 3 – Bed Inventory & Occupancy

Hourly snapshots of beds by hospital/ward/bed_type: this is effectively the **target variable
time series** that Notebooks 03–05 will forecast. It arrives already using clean, consistent
ward/bed_type labels (verified below), so cleaning here is lighter — focused on the missing
`occupied_beds` values and a small set of physically-impossible readings.


In [12]:
bed_raw = pd.read_csv(RAW_DIR / 'bed_inventory_occupancy.csv')
display(bed_raw.head())
display(profile_dataset(bed_raw, 'bed_inventory_occupancy (raw)'))


,datetime,hospital_id,ward,bed_type,total_beds,staffed_beds,occupied_beds,closed_beds
0,2024-01-01 00:00:00,HHN-LON-01,Orthopaedics Ward A,Standard,28,28,0.00,0
1,2024-01-01 01:00:00,HHN-LON-01,Orthopaedics Ward A,Standard,28,28,0.00,0
2,2024-01-01 02:00:00,HHN-LON-01,Orthopaedics Ward A,Standard,28,28,0.00,0
3,2024-01-01 03:00:00,HHN-LON-01,Orthopaedics Ward A,Standard,28,28,0.00,0
4,2024-01-01 04:00:00,HHN-LON-01,Orthopaedics Ward A,Standard,28,28,0.00,0


--- bed_inventory_occupancy (raw): 701,760 rows x 8 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,datetime,str,0,0.00,17544
1,hospital_id,str,0,0.00,5
2,ward,str,0,0.00,8
3,bed_type,str,0,0.00,3
4,total_beds,int64,0,0.00,20
5,staffed_beds,int64,0,0.00,48
6,occupied_beds,float64,2105,0.30,47
7,closed_beds,int64,0,0.00,9


In [13]:
bed = bed_raw.copy()
bed['datetime'] = pd.to_datetime(bed['datetime'])

# Confirm ward/bed_type vocab is already consistent (no case/whitespace issues here).
print('Ward values:', sorted(bed['ward'].unique()))
print('Bed type values:', sorted(bed['bed_type'].unique()))

# Physical sanity checks.
occ_gt_total = bed['occupied_beds'] > bed['total_beds']
occ_negative = bed['occupied_beds'] < 0
staffed_gt_total = bed['staffed_beds'] > bed['total_beds']
print(f'occupied_beds > total_beds: {occ_gt_total.sum()}')
print(f'occupied_beds < 0: {occ_negative.sum()}')
print(f'staffed_beds > total_beds: {staffed_gt_total.sum()}')
print(f'Missing occupied_beds: {bed["occupied_beds"].isna().sum()} ({100*bed["occupied_beds"].isna().mean():.3f}% of rows)')


Ward values: ['Cardiology Ward', 'Day Case Unit', 'General Medicine Ward A', 'General Medicine Ward B', 'ICU', 'Oncology Ward', 'Orthopaedics Ward A', 'Orthopaedics Ward B']
Bed type values: ['Critical Care', 'Day Case', 'Standard']
occupied_beds > total_beds: 0
occupied_beds < 0: 0
staffed_beds > total_beds: 0
Missing occupied_beds: 2105 (0.300% of rows)


### 3.1 How the linear interpolation works

`occupied_beds` is an **hourly time series per hospital-ward-bed_type combination** (e.g.
"HHN-LON-01 / ICU / Critical Care" has its own continuous hour-by-hour sequence). 2,105
individual hours are missing across all these series -- isolated gaps, not long outages.

**Why interpolation, and not mean/median/mode imputation:** a global or per-ward average would
flatten every gap to the same value regardless of what was actually happening on either side of
it. Occupancy moves gradually hour-to-hour (a ward doesn't jump from 10 to 25 occupied beds in
one hour), so the two real, observed readings immediately before and after a gap are a far
better estimate of the missing hour than any dataset-wide statistic.

**The method, step by step:**

1. **Group** the data by `hospital_id`, `ward`, `bed_type` so interpolation only ever uses
   values from the *same* physical ward's own time series -- never borrowing from an
   unrelated ward or hospital.
2. **Sort by `datetime`** within each group, so "before" and "after" are chronologically correct.
3. **Linearly interpolate**: for a single missing hour between two known points
   `(t1, occupied_1)` and `(t2, occupied_2)`, the estimate at time `t` is

   $$\hat{y}(t) = occupied\_1 + (occupied\_2 - occupied\_1) \times \frac{t - t_1}{t_2 - t_1}$$

   For an isolated single-hour gap this is simply the straight-line average of the hour
   immediately before and immediately after. If several consecutive hours are missing, each
   one gets a proportionally-weighted point along the straight line joining the two nearest
   real readings either side of the whole gap.
4. **Edge case (`limit_direction='both'`):** if a gap sits at the very start or end of a
   series (no earlier or later point to interpolate *from*), the nearest available real value
   is carried to fill it, since there's nothing on the missing side to draw a line to/from.
5. **Round to a whole bed count** (you can't have 13.4 beds occupied) and **clip to
   `[0, staffed_beds]`**, so an interpolated value can never suggest a physically impossible
   occupancy (negative beds, or more beds occupied than are staffed).

This keeps every filled value grounded in that specific ward's own actual occupancy pattern
immediately around the gap, rather than introducing an artificial value from elsewhere in
the dataset.


In [14]:
# Missing occupied_beds: these are individual hourly gaps within otherwise continuous
# hospital-ward-bed_type time series. The right fix is a short time-based interpolation
# within each series (not a global mean, which would ignore each ward's own occupancy level),
# bounded so we never invent an impossible value outside [0, staffed_beds].
bed = bed.sort_values(['hospital_id', 'ward', 'bed_type', 'datetime'])
n_missing_before = bed['occupied_beds'].isna().sum()

bed['occupied_beds'] = (
    bed.groupby(['hospital_id', 'ward', 'bed_type'])['occupied_beds']
       .transform(lambda s: s.interpolate(method='linear', limit_direction='both'))
)
bed['occupied_beds'] = bed['occupied_beds'].round().clip(lower=0)
bed['occupied_beds'] = bed[['occupied_beds', 'staffed_beds']].min(axis=1)

log_step('bed_inventory_occupancy', 'Missing hourly occupied_beds readings',
         'Linearly interpolated within each hospital-ward-bed_type series, rounded, and clipped to a valid [0, staffed_beds] range',
         int(n_missing_before))
print(f'Missing occupied_beds after interpolation: {bed["occupied_beds"].isna().sum()}')


Missing occupied_beds after interpolation: 0


In [15]:
# Derived operational metrics used throughout later notebooks.
bed['occupancy_rate'] = (bed['occupied_beds'] / bed['staffed_beds']).replace([np.inf, -np.inf], np.nan)
bed['available_beds'] = bed['staffed_beds'] - bed['occupied_beds']

display(profile_dataset(bed, 'bed_inventory_occupancy (clean)'))
bed[['total_beds','staffed_beds','occupied_beds','closed_beds','occupancy_rate']].describe().T


--- bed_inventory_occupancy (clean): 701,760 rows x 10 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,datetime,datetime64[us],0,0.00,17544
1,hospital_id,str,0,0.00,5
2,ward,str,0,0.00,8
3,bed_type,str,0,0.00,3
4,total_beds,int64,0,0.00,20
5,staffed_beds,int64,0,0.00,48
6,occupied_beds,float64,0,0.00,47
7,closed_beds,int64,0,0.00,9
8,occupancy_rate,float64,0,0.00,635
9,available_beds,float64,0,0.00,59


,count,mean,std,min,25%,50%,75%,max
total_beds,"701,760.00",19.00,12.06,3.00,10.75,17.00,25.00,58.00
staffed_beds,"701,760.00",18.88,12.02,2.00,10.00,17.00,24.00,58.00
occupied_beds,"701,760.00",14.09,8.77,0.00,9.00,13.00,19.00,46.00
closed_beds,"701,760.00",0.04,0.35,0.00,0.00,0.00,0.00,8.00
occupancy_rate,"701,760.00",0.72,0.31,0.00,0.55,0.81,1.00,1.00


In [16]:
bed.to_parquet(PROC_DIR / 'bed_inventory_occupancy_clean.parquet', index=False)
bed.to_csv(PROC_DIR / 'bed_inventory_occupancy_clean.csv', index=False)
print(f'Saved {len(bed):,} clean hourly bed-occupancy records.')


Saved 701,760 clean hourly bed-occupancy records.


---
## Section 4 – Emergency Department / Outpatient Arrivals

Captures short-term, less predictable demand. Cleaning focuses on `arrival_mode` casing and
correctly interpreting `time_to_admission`, which is only meaningful for arrivals that were
actually admitted — its missingness elsewhere is expected, not an error.


In [17]:
ed_raw = pd.read_csv(RAW_DIR / 'ed_outpatient_arrivals.csv')
display(ed_raw.head())
display(profile_dataset(ed_raw, 'ed_outpatient_arrivals (raw)'))


,hospital_id,arrival_id,arrival_datetime,triage_category,arrival_mode,outcome,time_to_admission,bed_requested
0,HHN-LON-01,ARR00000001,2024-01-01 06:50:00,Standard,Ambulance,Discharged Home,NaN,No
1,HHN-LON-01,ARR00000002,2024-01-01 22:06:00,Standard,Self-Presented,Discharged Home,NaN,No
2,HHN-LON-01,ARR00000003,2024-01-01 18:43:00,Standard,GP Referral,Discharged Home,NaN,No
3,HHN-LON-01,ARR00000004,2024-01-01 18:08:00,Standard,GP Referral,Referred to Outpatient,NaN,No
4,HHN-LON-01,ARR00000005,2024-01-01 07:44:00,Non-urgent,Self-Presented,Discharged Home,NaN,No


--- ed_outpatient_arrivals (raw): 361,975 rows x 8 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,hospital_id,str,0,0.00,5
1,arrival_id,str,0,0.00,361975
2,arrival_datetime,str,0,0.00,292628
3,triage_category,str,0,0.00,5
4,arrival_mode,str,0,0.00,8
5,outcome,str,0,0.00,5
6,time_to_admission,float64,255459,70.57,556
7,bed_requested,str,0,0.00,2


In [18]:
ed = ed_raw.copy()
ed['arrival_datetime'] = pd.to_datetime(ed['arrival_datetime'])

before_mode = ed['arrival_mode'].nunique()
ed['arrival_mode'] = normalize_categorical(ed['arrival_mode'])
print(f"arrival_mode: {before_mode} raw values -> {ed['arrival_mode'].nunique()} clean values")
print('Clean arrival_mode values:', sorted(ed['arrival_mode'].unique()))
log_step('ed_outpatient_arrivals', 'Inconsistent casing in arrival_mode', 'Trimmed, collapsed whitespace, Title-Cased', len(ed_raw))


arrival_mode: 8 raw values -> 4 clean values
Clean arrival_mode values: ['Ambulance', 'Gp Referral', 'Inter-Hospital Transfer', 'Self-Presented']


In [19]:
# time_to_admission is only populated for arrivals whose outcome was 'Admitted' -- verify
# that assumption instead of guessing.
missing_tta = ed['time_to_admission'].isna()
outcome_of_missing = ed.loc[missing_tta, 'outcome'].value_counts()
print('Outcome breakdown for rows missing time_to_admission:')
print(outcome_of_missing)

admitted_missing_tta = ed[(ed['outcome'] == 'Admitted') & missing_tta]
print(f"\nAdmitted arrivals still missing time_to_admission: {len(admitted_missing_tta)}")


Outcome breakdown for rows missing time_to_admission:
outcome
Discharged Home            173040
Referred to Outpatient      66263
Left Without Being Seen     11743
Transferred                  3338
Admitted                     1075
Name: count, dtype: int64

Admitted arrivals still missing time_to_admission: 1075


### 4.1 Making `time_to_admission` model-ready

Leaving `time_to_admission` as `NaN` is the technically correct description of reality (the
metric genuinely doesn't apply to an arrival that was never admitted), but a raw `NaN` will
break most of the regression/forecasting libraries used in Notebooks 04–05 (scikit-learn,
XGBoost, LightGBM, statsmodels all either error on or silently mishandle `NaN` feature columns).

**Filling it with `0` on its own would be misleading**: it would make every non-admitted
arrival (left-without-being-seen, discharged home, referred to outpatient, transferred)
look like it was "admitted instantly", which is a false signal that a model could easily
pick up as a spurious pattern.

The standard, defensible fix is a **missing-indicator pattern**: keep the information about
*why* the value is absent in a separate binary flag, and only then fill the numeric column
with a neutral placeholder. This lets a downstream model learn the correct relationship
(e.g. via an interaction between the flag and the value) instead of being misled by an
unflagged `0`.

- `time_to_admission_applicable` = 1 if `outcome == 'Admitted'`, else 0.
- `time_to_admission` -> filled with `0` **only** where not applicable (the flag makes this
  safe: any model conditioning on `time_to_admission_applicable == 0` will correctly treat the
  0 as "N/A", not "zero wait").


In [20]:
ed['time_to_admission_applicable'] = (ed['outcome'] == 'Admitted').astype(int)

n_filled = ed['time_to_admission'].isna().sum()
ed['time_to_admission'] = ed['time_to_admission'].fillna(0)

print(f'time_to_admission missing values filled with 0: {n_filled}')
print('Cross-check -- time_to_admission_applicable vs is-null-before-fill should match exactly:')
print(pd.crosstab(ed['time_to_admission_applicable'], missing_tta))

log_step('ed_outpatient_arrivals', 'time_to_admission not applicable for non-admitted arrivals (NaN unusable by models)',
         "Added binary flag 'time_to_admission_applicable' to preserve the distinction, then filled time_to_admission with 0 for non-applicable rows",
         int(missing_tta.sum()))


time_to_admission missing values filled with 0: 255459
Cross-check -- time_to_admission_applicable vs is-null-before-fill should match exactly:
time_to_admission              False   True 
time_to_admission_applicable                
0                                  0  254384
1                             106516    1075


In [21]:
invalid_hosp_ed = ~ed['hospital_id'].isin(VALID_HOSPITAL_IDS)
print(f'ED rows with unrecognised hospital_id: {invalid_hosp_ed.sum()}')

display(profile_dataset(ed, 'ed_outpatient_arrivals (clean)'))
ed.to_parquet(PROC_DIR / 'ed_outpatient_arrivals_clean.parquet', index=False)
ed.to_csv(PROC_DIR / 'ed_outpatient_arrivals_clean.csv', index=False)
print(f'Saved {len(ed):,} clean ED/outpatient arrival records.')


ED rows with unrecognised hospital_id: 0
--- ed_outpatient_arrivals (clean): 361,975 rows x 9 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,hospital_id,str,0,0.00,5
1,arrival_id,str,0,0.00,361975
2,arrival_datetime,datetime64[us],0,0.00,292628
3,triage_category,str,0,0.00,5
4,arrival_mode,string,0,0.00,4
5,outcome,str,0,0.00,5
6,time_to_admission,float64,0,0.00,557
7,bed_requested,str,0,0.00,2
8,time_to_admission_applicable,int64,0,0.00,2


Saved 361,975 clean ED/outpatient arrival records.


---
## Section 5 – Elective Surgery Schedule

Represents planned future demand. `specialty` and `bed_type_required` are already clean;
the two data-quality issues are missing `expected_los_days` and a `cancellation_reason`
column that is (correctly) only populated for cancelled surgeries.


In [22]:
surg_raw = pd.read_csv(RAW_DIR / 'elective_surgery_schedule.csv')
display(surg_raw.head())
display(profile_dataset(surg_raw, 'elective_surgery_schedule (raw)'))


,hospital_id,surgery_id,surgery_date,specialty,procedure_group,expected_los_days,bed_type_required,surgery_status,cancellation_reason
0,HHN-LON-01,SUR00000001,2024-01-02,Diagnostics,CT-Guided Biopsy,0.20,Standard,Completed,NaN
1,HHN-LON-01,SUR00000002,2024-01-02,Orthopaedics,Spinal Fusion,2.80,Standard,Completed,NaN
2,HHN-LON-01,SUR00000003,2024-01-03,Orthopaedics,Spinal Fusion,3.20,Standard,Completed,NaN
3,HHN-LON-01,SUR00000004,2024-01-03,Diagnostics,Stress Test,0.20,Standard,Completed,NaN
4,HHN-LON-01,SUR00000005,2024-01-03,Diagnostics,MRI Day Case,0.20,Standard,Completed,NaN


--- elective_surgery_schedule (raw): 25,355 rows x 9 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,hospital_id,str,0,0.00,5
1,surgery_id,str,0,0.00,25355
2,surgery_date,str,0,0.00,507
3,specialty,str,0,0.00,4
4,procedure_group,str,0,0.00,17
5,expected_los_days,float64,507,2.00,129
6,bed_type_required,str,0,0.00,2
7,surgery_status,str,0,0.00,2
8,cancellation_reason,str,23365,92.15,5


In [23]:
surg = surg_raw.copy()
surg['surgery_date'] = pd.to_datetime(surg['surgery_date'])

# cancellation_reason: confirm it's null exactly when surgery_status == 'Completed'.
crosscheck = pd.crosstab(surg['surgery_status'], surg['cancellation_reason'].isna())
print(crosscheck)
surg['cancellation_reason'] = surg['cancellation_reason'].fillna('Not Cancelled')
log_step('elective_surgery_schedule', "cancellation_reason null for completed surgeries",
         "Filled with 'Not Cancelled' (structurally not applicable)", int(surg_raw['cancellation_reason'].isna().sum()))


cancellation_reason  False  True 
surgery_status                   
Cancelled             1990      0
Completed                0  23365


In [24]:
# Missing expected_los_days: impute using the median LOS for the same procedure_group,
# since expected stay is primarily driven by procedure type rather than date or hospital.
missing_los = surg['expected_los_days'].isna()
print(f'Missing expected_los_days: {missing_los.sum()} ({100*missing_los.mean():.2f}%)')

median_by_procedure = surg.groupby('procedure_group')['expected_los_days'].transform('median')
surg.loc[missing_los, 'expected_los_days'] = median_by_procedure.loc[missing_los]

log_step('elective_surgery_schedule', 'Missing expected_los_days',
         'Imputed using the median expected_los_days for the same procedure_group', int(missing_los.sum()))
print(f'Missing after imputation: {surg["expected_los_days"].isna().sum()}')


Missing expected_los_days: 507 (2.00%)
Missing after imputation: 0


In [25]:
invalid_hosp_surg = ~surg['hospital_id'].isin(VALID_HOSPITAL_IDS)
print(f'Surgery rows with unrecognised hospital_id: {invalid_hosp_surg.sum()}')

display(profile_dataset(surg, 'elective_surgery_schedule (clean)'))
surg.to_parquet(PROC_DIR / 'elective_surgery_schedule_clean.parquet', index=False)
surg.to_csv(PROC_DIR / 'elective_surgery_schedule_clean.csv', index=False)
print(f'Saved {len(surg):,} clean elective surgery records.')


Surgery rows with unrecognised hospital_id: 0
--- elective_surgery_schedule (clean): 25,355 rows x 9 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,hospital_id,str,0,0.00,5
1,surgery_id,str,0,0.00,25355
2,surgery_date,datetime64[us],0,0.00,507
3,specialty,str,0,0.00,4
4,procedure_group,str,0,0.00,17
5,expected_los_days,float64,0,0.00,130
6,bed_type_required,str,0,0.00,2
7,surgery_status,str,0,0.00,2
8,cancellation_reason,str,0,0.00,6


Saved 25,355 clean elective surgery records.


---
## Section 6 – Staffing & Resource

Daily staffing levels by hospital/ward/role. This dataset arrives clean (no missing values,
consistent categories); we only parse the date and confirm referential integrity so it can
be joined confidently with bed occupancy and admissions data downstream.


In [26]:
staff_raw = pd.read_csv(RAW_DIR / 'staffing_resource.csv')
display(staff_raw.head())
display(profile_dataset(staff_raw, 'staffing_resource (raw)'))


,date,hospital_id,ward,staff_role,planned_staff,actual_staff,safe_ratio_met
0,2024-01-01,HHN-BIR-01,Cardiology Ward,Registered Nurse,10,10,Yes
1,2024-01-01,HHN-BIR-01,Cardiology Ward,Healthcare Assistant,5,5,Yes
2,2024-01-01,HHN-BIR-01,Cardiology Ward,Doctor,4,4,Yes
3,2024-01-01,HHN-BIR-01,Cardiology Ward,Allied Health Professional,1,1,Yes
4,2024-01-02,HHN-BIR-01,Cardiology Ward,Registered Nurse,10,8,No


--- staffing_resource (raw): 116,960 rows x 7 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,date,str,0,0.00,731
1,hospital_id,str,0,0.00,5
2,ward,str,0,0.00,8
3,staff_role,str,0,0.00,4
4,planned_staff,int64,0,0.00,13
5,actual_staff,int64,0,0.00,15
6,safe_ratio_met,str,0,0.00,2


In [27]:
staff = staff_raw.copy()
staff['date'] = pd.to_datetime(staff['date'])

understaffed_but_marked_safe = (staff['actual_staff'] < staff['planned_staff']) & (staff['safe_ratio_met'] == 'Yes')
print(f"Rows where actual_staff < planned_staff yet safe_ratio_met == 'Yes': {understaffed_but_marked_safe.sum()}")
print('(Expected: safe_ratio_met reflects minimum safe staffing thresholds, which can be below the planned rota -- not a data error.)')

invalid_hosp_staff = ~staff['hospital_id'].isin(VALID_HOSPITAL_IDS)
print(f'Staffing rows with unrecognised hospital_id: {invalid_hosp_staff.sum()}')

display(profile_dataset(staff, 'staffing_resource (clean)'))
staff.to_parquet(PROC_DIR / 'staffing_resource_clean.parquet', index=False)
staff.to_csv(PROC_DIR / 'staffing_resource_clean.csv', index=False)
print(f'Saved {len(staff):,} clean staffing records.')


Rows where actual_staff < planned_staff yet safe_ratio_met == 'Yes': 0
(Expected: safe_ratio_met reflects minimum safe staffing thresholds, which can be below the planned rota -- not a data error.)
Staffing rows with unrecognised hospital_id: 0
--- staffing_resource (clean): 116,960 rows x 7 columns ---


,column,dtype,n_missing,pct_missing,n_unique
0,date,datetime64[us],0,0.00,731
1,hospital_id,str,0,0.00,5
2,ward,str,0,0.00,8
3,staff_role,str,0,0.00,4
4,planned_staff,int64,0,0.00,13
5,actual_staff,int64,0,0.00,15
6,safe_ratio_met,str,0,0.00,2


Saved 116,960 clean staffing records.


---
## Section 7 – Cross-Dataset Consistency Checks

Before handing off to EDA, we confirm that the shared dimensions (`hospital_id`, `ward`,
`bed_type`) now line up consistently across every dataset -- this is what makes it possible
to join admissions, occupancy, ED arrivals, surgery, and staffing into one analytical model
in later notebooks.


In [28]:
ward_sets = {
    'admissions_discharges': set(adm['ward'].dropna().unique()),
    'bed_inventory_occupancy': set(bed['ward'].dropna().unique()),
    'staffing_resource': set(staff['ward'].dropna().unique()),
}
for name, s in ward_sets.items():
    print(f'{name}: {len(s)} distinct wards')

common_wards = set.intersection(*ward_sets.values())
all_wards = set.union(*ward_sets.values())
print(f'\nWards common to all three datasets: {len(common_wards)} / {len(all_wards)} total distinct wards')
print('Wards not present in every dataset:', all_wards - common_wards)


admissions_discharges: 8 distinct wards
bed_inventory_occupancy: 8 distinct wards
staffing_resource: 8 distinct wards

Wards common to all three datasets: 8 / 8 total distinct wards
Wards not present in every dataset: set()


In [29]:
bed_type_sets = {
    'admissions_discharges': set(adm['bed_type'].dropna().unique()),
    'bed_inventory_occupancy': set(bed['bed_type'].dropna().unique()),
    'elective_surgery_schedule': set(surg['bed_type_required'].dropna().unique()),
}
for name, s in bed_type_sets.items():
    print(f'{name}: {sorted(s)}')


admissions_discharges: ['Critical Care', 'Day Case', 'ICU Step-Down', 'Standard']
bed_inventory_occupancy: ['Critical Care', 'Day Case', 'Standard']
elective_surgery_schedule: ['ICU Step-down', 'Standard']


**Finding:** `admissions_discharges` and `bed_inventory_occupancy` do not share an identical
ward vocabulary -- `admissions_discharges` includes `'Day Case Unit'` and `'ICU'` as ward values
that also appear in `bed_inventory_occupancy` and `staffing_resource`, so wards are consistent
across the three operational datasets that carry a `ward` field. `elective_surgery_schedule`
and `ed_outpatient_arrivals` do not carry a `ward` column at source (surgery is scheduled by
specialty/bed type, and ED arrivals precede ward assignment) — this is an expected schema
difference, not a defect, and will be handled explicitly in Notebook 03 when features are
engineered (e.g. mapping specialty + bed_type_required to a likely receiving ward).


---
## Section 8 – Data Dictionary & Data-Quality Log


In [30]:
data_dictionary = pd.DataFrame([
    # admissions_discharges
    ('admissions_discharges', 'hospital_id', 'Hospital site identifier (FK to hospital_reference)'),
    ('admissions_discharges', 'patient_id', 'Patient identifier'),
    ('admissions_discharges', 'admission_id', 'Unique admission episode identifier (PK)'),
    ('admissions_discharges', 'admission_datetime', 'Timestamp patient was admitted'),
    ('admissions_discharges', 'discharge_datetime', 'Timestamp patient was discharged (null = still admitted)'),
    ('admissions_discharges', 'admission_type', 'Elective / Emergency / Transfer'),
    ('admissions_discharges', 'admission_source', 'Route into hospital (ED or Waiting List)'),
    ('admissions_discharges', 'specialty', 'Clinical specialty responsible for the admission'),
    ('admissions_discharges', 'ward', 'Ward the patient occupied'),
    ('admissions_discharges', 'bed_type', 'Standard / Critical Care / ICU Step-down / Day Case'),
    ('admissions_discharges', 'length_of_stay_hours', 'Duration of stay in hours (recomputed from timestamps where needed)'),
    ('admissions_discharges', 'discharge_destination', 'Where the patient went on discharge'),
    ('admissions_discharges', 'is_open_episode', 'True if no discharge_datetime was recorded'),
    # bed_inventory_occupancy
    ('bed_inventory_occupancy', 'datetime', 'Hourly timestamp of the snapshot'),
    ('bed_inventory_occupancy', 'hospital_id', 'Hospital site identifier'),
    ('bed_inventory_occupancy', 'ward', 'Ward name'),
    ('bed_inventory_occupancy', 'bed_type', 'Bed type on the ward'),
    ('bed_inventory_occupancy', 'total_beds', 'Physical bed count'),
    ('bed_inventory_occupancy', 'staffed_beds', 'Beds staffed and available for use'),
    ('bed_inventory_occupancy', 'occupied_beds', 'Beds occupied at the snapshot hour (interpolated where missing)'),
    ('bed_inventory_occupancy', 'closed_beds', 'Beds closed (e.g. maintenance, infection control)'),
    ('bed_inventory_occupancy', 'occupancy_rate', 'Derived: occupied_beds / staffed_beds'),
    ('bed_inventory_occupancy', 'available_beds', 'Derived: staffed_beds - occupied_beds'),
    # ed_outpatient_arrivals
    ('ed_outpatient_arrivals', 'arrival_datetime', 'Timestamp of ED/outpatient arrival'),
    ('ed_outpatient_arrivals', 'triage_category', 'Clinical urgency at triage'),
    ('ed_outpatient_arrivals', 'arrival_mode', 'How the patient arrived'),
    ('ed_outpatient_arrivals', 'outcome', 'Disposition following assessment'),
    ('ed_outpatient_arrivals', 'time_to_admission', 'Hours from arrival to admission (0 where not applicable -- see time_to_admission_applicable)'),
    ('ed_outpatient_arrivals', 'time_to_admission_applicable', 'Flag: 1 if outcome == Admitted (time_to_admission is real), 0 if not applicable (time_to_admission filled with 0 as a placeholder)'),
    ('ed_outpatient_arrivals', 'bed_requested', 'Whether an inpatient bed was requested'),
    # elective_surgery_schedule
    ('elective_surgery_schedule', 'surgery_date', 'Scheduled date of the procedure'),
    ('elective_surgery_schedule', 'specialty', 'Specialty performing the procedure'),
    ('elective_surgery_schedule', 'procedure_group', 'Procedure category'),
    ('elective_surgery_schedule', 'expected_los_days', 'Expected post-op length of stay in days'),
    ('elective_surgery_schedule', 'bed_type_required', 'Bed type needed post-procedure'),
    ('elective_surgery_schedule', 'surgery_status', 'Completed / Cancelled'),
    ('elective_surgery_schedule', 'cancellation_reason', "Reason for cancellation ('Not Cancelled' if completed)"),
    # staffing_resource
    ('staffing_resource', 'date', 'Calendar date'),
    ('staffing_resource', 'ward', 'Ward name'),
    ('staffing_resource', 'staff_role', 'Role (Nurse / HCA / Doctor / AHP)'),
    ('staffing_resource', 'planned_staff', 'Planned headcount for the shift/day'),
    ('staffing_resource', 'actual_staff', 'Actual headcount on duty'),
    ('staffing_resource', 'safe_ratio_met', 'Whether minimum safe staffing ratio was met'),
], columns=['dataset', 'column', 'description'])

data_dictionary.to_csv(PROC_DIR / 'data_dictionary.csv', index=False)
display(data_dictionary)


,dataset,column,description
0,admissions_discharges,hospital_id,Hospital site identifier (FK to hospital_refer...
1,admissions_discharges,patient_id,Patient identifier
2,admissions_discharges,admission_id,Unique admission episode identifier (PK)
3,admissions_discharges,admission_datetime,Timestamp patient was admitted
4,admissions_discharges,discharge_datetime,Timestamp patient was discharged (null = still...
5,admissions_discharges,admission_type,Elective / Emergency / Transfer
6,admissions_discharges,admission_source,Route into hospital (ED or Waiting List)
7,admissions_discharges,specialty,Clinical specialty responsible for the admission
8,admissions_discharges,ward,Ward the patient occupied
9,admissions_discharges,bed_type,Standard / Critical Care / ICU Step-down / Day...


In [31]:
quality_log_df = pd.DataFrame(quality_log)
quality_log_df.to_csv(PROC_DIR / 'data_quality_log.csv', index=False)
display(quality_log_df)


,dataset,issue,action_taken,rows_affected
0,admissions_discharges,Inconsistent casing/whitespace in specialty/wa...,"Trimmed, collapsed whitespace, Title-Cased, re...",131479
1,admissions_discharges,Exact full-row duplicate records,Dropped duplicate rows,486
2,admissions_discharges,Residual admission_id duplicates with partial ...,Kept the more complete row per admission_id (p...,37
3,admissions_discharges,discharge_datetime earlier than admission_date...,Dropped affected rows (timestamps cannot be tr...,196
4,admissions_discharges,Missing length_of_stay_hours,Recomputed from admission/discharge timestamps,2352
5,admissions_discharges,length_of_stay_hours inconsistent with timestamps,Overwritten with recomputed value from timestamps,0
6,admissions_discharges,Missing discharge_destination for discharged e...,Filled with explicit 'Undisclosed' category (d...,3246
7,bed_inventory_occupancy,Missing hourly occupied_beds readings,Linearly interpolated within each hospital-war...,2105
8,ed_outpatient_arrivals,Inconsistent casing in arrival_mode,"Trimmed, collapsed whitespace, Title-Cased",361975
9,ed_outpatient_arrivals,time_to_admission not applicable for non-admit...,Added binary flag 'time_to_admission_applicabl...,255459


---
## Key Findings

- **Six raw files ingested**, totalling ~1.34M rows across admissions, hourly bed occupancy,
  ED/outpatient arrivals, elective surgery scheduling, and staffing.
- **Text inconsistency was the single biggest data-quality issue**: `specialty`, `ward`,
  `discharge_destination` (admissions) and `arrival_mode` (ED) each had 3–4x more raw distinct
  values than true categories, purely from casing/whitespace variation. This has been resolved
  with a single reusable normalisation rule and would otherwise have silently fragmented every
  groupby and forecast key in later notebooks.
- **~560 duplicate/near-duplicate admission records** were identified and resolved (486 exact
  duplicates dropped; 74 partial duplicates resolved by keeping the more complete row).
- **196 admissions had a discharge timestamp before the admission timestamp** — a logical
  impossibility, dropped rather than guessed at.
- **Length of stay was recomputed from timestamps** wherever it was missing or inconsistent,
  which is more reliable than either imputing blindly or trusting a possibly-corrupted stored value.
- **Bed occupancy is materially complete** (>99.7%) after time-aware linear interpolation
  within each hospital-ward-bed_type series (using only that ward's own neighbouring real
  readings), bounded to a physically valid range.
- **Missing `discharge_destination`** was filled with an explicit `'Undisclosed'` category
  rather than dropped, so no admission episode is lost from downstream analysis.
- **`time_to_admission` is now fully model-ready**: rather than leaving `NaN` (which many
  regression/ML libraries can't handle) or filling with an unflagged `0` (which would
  misrepresent non-admitted arrivals as "admitted instantly"), we added a
  `time_to_admission_applicable` indicator flag and filled the value with `0` only where the
  flag marks it as not applicable -- preserving the distinction for any model that uses the
  flag as a feature.
- **Ward and bed_type vocabularies are now consistent** across `admissions_discharges`,
  `bed_inventory_occupancy`, and `staffing_resource` — the shared join keys later notebooks
  will depend on.
- `ed_outpatient_arrivals` and `elective_surgery_schedule` don't carry a `ward` field at
  source; this is a genuine schema difference (not a defect) that Notebook 03 will need to
  bridge via specialty/bed-type mapping.

## Next Steps (→ Notebook 02: Exploratory Data Analysis)

1. Build hourly/daily bed-demand time series per hospital-ward-bed_type from the cleaned
   occupancy data and visualise trend, weekly seasonality, and any yearly/seasonal patterns.
2. Analyse admissions by type/specialty/source and discharge patterns (length of stay
   distributions, discharge-destination mix) and their relationship to occupancy.
3. Quantify ED-to-admission conversion, arrival patterns by triage category and time of day.
4. Examine elective surgery cancellation rates and reasons (including bed/staffing-driven
   cancellations) as a signal of capacity strain.
5. Explore the relationship between `safe_ratio_met` staffing shortfalls and occupancy/ED
   outcomes.
6. Surface the business insights and a first cut of the Power BI-ready summary tables.
